# ThreatHunter QLoRA Fine-Tuning
Run this notebook in Google Colab with an A100 GPU.

In [ ]:
# ============================================================
# CELL 1 — Setup
# ============================================================
!pip install transformers peft bitsandbytes trl accelerate datasets -q

from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/huntgpt/data', exist_ok=True)
os.makedirs('/content/drive/MyDrive/huntgpt/models/threathunter', exist_ok=True)

In [ ]:
# ============================================================
# CELL 2 — Clone labelled OPTC data
# ============================================================
!git clone https://gitlab.inria.fr/fmajorcz/a_new_hope_for_darpa_optc.git /content/optc_labels

In [ ]:
# ============================================================
# CELL 3 — Load events and ATT&CK corpus
# ============================================================
import json, gzip

def load_events(path):
    with gzip.open(path, 'rt') as f:
        return [json.loads(line) for line in f if line.strip()]

events = load_events('/content/optc_labels/labelling/host/ground_truths/ground_truth_corrected_all_evts.json.gz')
print(f"Total events: {len(events)}")

# Load your ATT&CK corpus for procedure_examples augmentation
import urllib.request
attack_url = "https://raw.githubusercontent.com/spoorthi2615/HuntGPT/main/data/attack_corpus/techniques.json"
with urllib.request.urlopen(attack_url) as resp:
    attack_techniques = json.loads(resp.read())
attack_by_id = {t['technique_id']: t for t in attack_techniques}
print(f"ATT&CK techniques loaded: {len(attack_by_id)}")

In [ ]:
# ============================================================
# CELL 4 — Final verified TECHNIQUE_MAP
# ============================================================
TECHNIQUE_MAP = {
    "powershell.exe": "T1059.001",
    "cmd.exe": "T1059.003",
    "cscript.exe": "T1059.005",
    "ping.exe": "T1018",
    "netstat.exe": "T1016",
    "whoami.exe": "T1033",
    "eventvwr.exe": "T1548.002",
    "schtasks.exe": "T1053.005",
    "net.exe": "T1136.001",
    "net1.exe": "T1136.001",
    "reg.exe": "T1112",
    "ckfgw.exe": "T1036",
    "f.exe": "T1036",
    "biguwcmnsucig.exe": "T1027",
    "filetransfer1000.exe": "T1048",
    "movingonup.exe": "T1048",
}

In [ ]:
# ============================================================
# CELL 5 — Build training pairs from real OPTC events
# ============================================================
training_pairs = []
unmapped_count = 0

for e in events:
    img = e.get('properties', {}).get('image_path', '').split('\\')[-1].lower()
    technique_id = TECHNIQUE_MAP.get(img)
    if technique_id is None:
        unmapped_count += 1
        continue

    technique_info = attack_by_id.get(technique_id, {})
    log_summary = (
        f"timestamp={e.get('timestamp')} "
        f"host={e.get('hostname')} "
        f"process={img} "
        f"ppid={e.get('ppid')} "
        f"cmd={e.get('properties', {}).get('command_line', '')[:200]} "
        f"user={e.get('properties', {}).get('user', '')}"
    )

    training_pairs.append({
        "log": log_summary,
        "technique_id": technique_id,
        "technique_name": technique_info.get('technique_name', technique_id),
        "source": "optc_real"
    })

print(f"Real training pairs built: {len(training_pairs)}")
print(f"Unmapped events skipped: {unmapped_count}")

In [ ]:
# ============================================================
# CELL 6 — Augment with ATT&CK procedure examples (for diversity)
# ============================================================
for tid, info in attack_by_id.items():
    if tid in TECHNIQUE_MAP.values():
        for proc_example in info.get('procedure_examples', [])[:3]:
            training_pairs.append({
                "log": f"procedure_description={proc_example}",
                "technique_id": tid,
                "technique_name": info.get('technique_name', tid),
                "source": "attack_augmented"
            })

print(f"Total training pairs after augmentation: {len(training_pairs)}")

In [ ]:
# ============================================================
# CELL 7 — Downsample, Dedupe and split, save to Drive
# ============================================================
import random
from collections import defaultdict, Counter
random.seed(42)

seen = set()
unique_pairs = []
for p in training_pairs:
    key = p['log']
    if key not in seen:
        seen.add(key)
        unique_pairs.append(p)

by_technique = defaultdict(list)
for p in unique_pairs:
    by_technique[p['technique_id']].append(p)

CAP = 500  # max examples per technique
balanced_pairs = []
for tid, examples in by_technique.items():
    if len(examples) > CAP:
        balanced_pairs.extend(random.sample(examples, CAP))
    else:
        balanced_pairs.extend(examples)

random.shuffle(balanced_pairs)
print(f"\nBalanced total: {len(balanced_pairs)}")
dist = Counter(p['technique_id'] for p in balanced_pairs)
for tid, count in dist.most_common():
    print(f"{tid}: {count}")

n = len(balanced_pairs)
train = balanced_pairs[:int(n*0.8)]
val = balanced_pairs[int(n*0.8):int(n*0.9)]
test = balanced_pairs[int(n*0.9):]

output = {"train": train, "val": val, "test": test}
with open('/content/drive/MyDrive/huntgpt/data/training_pairs_real.json', 'w') as f:
    json.dump(output, f, indent=2)

print(f"Saved. Train={len(train)} Val={len(val)} Test={len(test)}")

In [ ]:
# ============================================================
# CELL 8 — Load base model with 4-bit quantization
# ============================================================
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

In [ ]:
# ============================================================
# CELL 9 — Format training data as instruction pairs
# ============================================================
def format_example(pair):
    prompt = f"""Given this security log, identify the MITRE ATT&CK technique.
Respond ONLY with valid JSON: {{"technique_id": "T####.###", "technique_name": "...", "hypothesis": "..."}}

Log: {pair['log']}"""
    response = json.dumps({
        "technique_id": pair['technique_id'],
        "technique_name": pair['technique_name'],
        "hypothesis": f"Behavior consistent with {pair['technique_name']}."
    })
    return f"<s>[INST] {prompt} [/INST] {response}</s>"

with open('/content/drive/MyDrive/huntgpt/data/training_pairs_real.json') as f:
    data = json.load(f)

train_texts = [format_example(p) for p in data['train']]
val_texts = [format_example(p) for p in data['val']]

from datasets import Dataset
train_ds = Dataset.from_dict({"text": train_texts})
val_ds = Dataset.from_dict({"text": val_texts})
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")
print(train_texts[0])  # sanity check the format

In [ ]:
# ============================================================
# CELL 10 — LoRA config and SFTTrainer
# ============================================================
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

sft_config = SFTConfig(
    output_dir="/content/threathunter_checkpoint",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    bf16=True,
    max_seq_length=512,
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    peft_config=lora_config,
)

trainer.train()

In [ ]:
# ============================================================
# CELL 11 — Save adapter to Drive
# ============================================================
trainer.model.save_pretrained('/content/drive/MyDrive/huntgpt/models/threathunter/')
tokenizer.save_pretrained('/content/drive/MyDrive/huntgpt/models/threathunter/')
print("Saved.")

In [ ]:
# ============================================================
# CELL 12 — Sanity check
# ============================================================
test_log = data['test'][0]
prompt = f"""Given this security log, identify the MITRE ATT&CK technique.
Respond ONLY with valid JSON: {{"technique_id": "T####.###", "technique_name": "...", "hypothesis": "..."}}

Log: {test_log['log']}"""

inputs = tokenizer(f"<s>[INST] {prompt} [/INST]", return_tensors="pt").to(model.device)
output = model.generate(**inputs, max_new_tokens=150, temperature=0.1)
print("Expected technique:", test_log['technique_id'])
print("Model output:", tokenizer.decode(output[0], skip_special_tokens=True))